In [ ]:
# BLOQUE 1. INSTALACIÓN Y CONFIGURACIÓN

!pip -q install --upgrade pip
!pip -q install yfinance pandas numpy requests feedparser newspaper4k lxml_html_clean beautifulsoup4 tqdm openpyxl

import os
import re
import time
import html
import json
import math
import random
import warnings
import requests
import feedparser
import numpy as np
import pandas as pd
import yfinance as yf

from tqdm import tqdm
from datetime import datetime, timedelta
from urllib.parse import quote, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

from newspaper import Article

warnings.filterwarnings("ignore")

# RUTAS

BASE_DIR = "/content/drive/MyDrive/EAFIT/TESIS"
os.makedirs(BASE_DIR, exist_ok=True)

PRICE_PATH = os.path.join(BASE_DIR, "precios_colombia.csv")
NEWS_RAW_PATH = os.path.join(BASE_DIR, "news_raw_colombia.csv")
NEWS_DAILY_PATH = os.path.join(BASE_DIR, "news_sentiment_daily_colombia.csv")
DATASET_BASE_PATH = os.path.join(BASE_DIR, "dataset_modelo_base.csv")
SUMMARY_PATH = os.path.join(BASE_DIR, "resumen_etapas.json")

# PARÁMETROS
START_DATE = "2021-01-14"
END_DATE   = "2025-12-31"
TARGET_HORIZON = 1

# TICKERS
TICKER_MAP = {
    "EC": "EC",
    "ARG": "GRUPOARGOS.CL",
    "CIBEST": "CIBEST.CL",
    "NCH": "NUTRESA.CL"
}

# QUERIES DE NOTICIAS
STOCK_QUERIES = {
    "EC": [
        '"Ecopetrol"',
        '"acción Ecopetrol"',
        '"acciones Ecopetrol"',
        '"Ecopetrol resultados"',
        '"Ecopetrol bolsa"'
    ],
    "ARG": [
        '"Grupo Argos"',
        '"acción Grupo Argos"',
        '"acciones Grupo Argos"',
        '"Grupo Argos resultados"',
        '"Grupo Argos bolsa"'
    ],
    "CIBEST": [
        '"Grupo Cibest"',
        '"Cibest"',
        '"Bancolombia"',
        '"acción Bancolombia"',
        '"acciones Bancolombia"',
        '"Bancolombia resultados"'
    ],
    "NCH": [
        '"Nutresa"',
        '"Grupo Nutresa"',
        '"acción Nutresa"',
        '"acciones Nutresa"',
        '"Nutresa resultados"'
    ]
}

COLOMBIAN_DOMAINS = [
    "larepublica.co",
    "portafolio.co",
    "elcolombiano.com",
    "eltiempo.com",
    "elespectador.com",
    "semana.com",
    "valoraanalitik.com",
    "forbes.co",
    "dinero.com",
    "infobae.com"
]

GOOGLE_NEWS_RSS = "https://news.google.com/rss/search"

print("Configuración lista.")
print("Base dir:", BASE_DIR)
print("Archivos de salida:")
print("-", PRICE_PATH)
print("-", NEWS_RAW_PATH)
print("-", NEWS_DAILY_PATH)
print("-", DATASET_BASE_PATH)
print("-", SUMMARY_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Configuración lista.
Base dir: /content/drive/MyDrive/EAFIT/TESIS
Archivos de salida:
- /content/drive/MyDrive/EAFIT/TESIS/precios_colombia.csv
- /content/drive/MyDrive/EAFIT/TESIS/news_raw_colombia.csv
- /content/drive/MyDrive/EAFIT/TESIS/news_sentiment_daily_colombia.csv
- /content/drive/MyDrive/EAFIT/TESIS/dataset_modelo_base.csv
- /content/drive/MyDrive/EAFIT/TESIS/resumen_etapas.json


In [ ]:
# BLOQUE 2. FUNCIONES AUXILIARES

def normalize_text(text):
    if text is None:
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_domain(url):
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except:
        return ""

def is_priority_source(url):
    u = (url or "").lower()
    return any(dom in u for dom in COLOMBIAN_DOMAINS)

def chunk_date_ranges(start_date, end_date, step_days=7):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    chunks = []
    cur = start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=step_days - 1), end)
        chunks.append((cur.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        cur = chunk_end + timedelta(days=1)
    return chunks

def build_google_news_rss_url(query, date_from, date_to, lang="es-419", country="CO"):
    search_query = f"{query} after:{date_from} before:{date_to}"
    params = {
        "q": search_query,
        "hl": lang,
        "gl": country,
        "ceid": f"{country}:{lang}",
    }
    qs = "&".join([f"{k}={quote(str(v))}" for k, v in params.items()])
    return f"{GOOGLE_NEWS_RSS}?{qs}"

def resolve_final_url(url, timeout=20):
    try:
        r = requests.get(
            url,
            timeout=timeout,
            headers={"User-Agent": "Mozilla/5.0"},
            allow_redirects=True
        )
        return r.url
    except:
        return url

def parse_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()

        title = normalize_text(article.title)
        text = normalize_text(article.text)
        publish_date = article.publish_date

        if publish_date is not None:
            publish_date = pd.to_datetime(publish_date, errors="coerce")
            if pd.notnull(publish_date):
                publish_date = publish_date.strftime("%Y-%m-%d")
            else:
                publish_date = None

        return {
            "title_extracted": title,
            "text_extracted": text,
            "publish_date_extracted": publish_date
        }
    except:
        return {
            "title_extracted": "",
            "text_extracted": "",
            "publish_date_extracted": None
        }

def classify_sentiment(text):
    if not text or len(text.strip()) < 20:
        return "NEU", 0

    try:
        pred = sentiment_analyzer.predict(text)
        label = pred.output

        if label == "POS":
            return "POS", 1
        elif label == "NEG":
            return "NEG", -1
        else:
            return "NEU", 0
    except:
        return "NEU", 0

def relevance_filter(stock, title, summary, article_text):
    text = " ".join([title or "", summary or "", article_text or ""]).lower()

    rules = {
        "EC": ["ecopetrol"],
        "ARG": ["grupo argos", "argos"],
        "CIBEST": ["grupo cibest", "cibest", "bancolombia"],
        "NCH": ["nutresa", "grupo nutresa"],
    }

    keywords = rules.get(stock, [])
    return any(k in text for k in keywords)

def evaluate_daily_label(scores):
    if len(scores) == 0:
        return 0
    s = np.sum(scores)
    if s > 0:
        return 1
    elif s < 0:
        return -1
    return 0

In [ ]:
# BLOQUE 3. PRECIOS Y TARGET

all_prices = []

for stock_name, yf_ticker in TICKER_MAP.items():
    print(f"Descargando {stock_name} -> {yf_ticker}")

    try:
        tmp = yf.download(
            yf_ticker,
            start=START_DATE,
            end=(pd.to_datetime(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
            auto_adjust=False,
            progress=False
        )

        if tmp.empty:
            print(f"[WARNING] Sin datos para {stock_name}")
            continue

        # Si viene con MultiIndex, aplanar columnas
        if isinstance(tmp.columns, pd.MultiIndex):
            tmp.columns = [col[0] if isinstance(col, tuple) else col for col in tmp.columns]

        tmp = tmp.reset_index()

        print("Columnas descargadas:", tmp.columns.tolist())

        # Identificar columna fecha
        if "Date" in tmp.columns:
            tmp = tmp.rename(columns={"Date": "fecha"})
        elif "index" in tmp.columns:
            tmp = tmp.rename(columns={"index": "fecha"})

        # Elegir columna de cierre
        if "Adj Close" in tmp.columns:
            close_col = "Adj Close"
        elif "Close" in tmp.columns:
            close_col = "Close"
        else:
            print(f"[WARNING] No se encontró columna de cierre para {stock_name}")
            continue

        rename_map = {
            close_col: "close",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Volume": "volume"
        }

        tmp = tmp.rename(columns=rename_map)
        tmp["stock"] = stock_name

        keep_cols = ["fecha", "stock", "open", "high", "low", "close", "volume"]
        keep_cols = [c for c in keep_cols if c in tmp.columns]
        tmp = tmp[keep_cols].copy()

        if "close" not in tmp.columns:
            print(f"[WARNING] No se pudo construir 'close' para {stock_name}")
            continue

        tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")
        tmp = tmp.dropna(subset=["fecha", "close"]).copy()
        tmp["fecha"] = tmp["fecha"].dt.strftime("%Y-%m-%d")

        tmp = tmp.sort_values("fecha").reset_index(drop=True)

        # Features base
        tmp["return_1d"] = tmp["close"].pct_change()
        tmp["volatility_5"] = tmp["return_1d"].rolling(5).std()
        tmp["ma_5"] = tmp["close"].rolling(5).mean()
        tmp["ma_10"] = tmp["close"].rolling(10).mean()

        # Target binario
        tmp["target"] = (tmp["close"].shift(-TARGET_HORIZON) > tmp["close"]).astype(int)

        all_prices.append(tmp)

    except Exception as e:
        print(f"[ERROR] {stock_name}: {e}")

if len(all_prices) == 0:
    raise ValueError("No se descargó información de ningún ticker. Revisa TICKER_MAP.")

prices_df = pd.concat(all_prices, ignore_index=True)
prices_df = prices_df.sort_values(["stock", "fecha"]).reset_index(drop=True)

prices_df = prices_df.dropna(subset=["close"]).copy()
prices_df = prices_df.dropna(subset=["return_1d", "volatility_5", "ma_5", "ma_10"]).copy()

prices_df.to_csv(PRICE_PATH, index=False, encoding="utf-8-sig")

print("\nPrecios guardados en:", PRICE_PATH)
print("Shape precios:", prices_df.shape)
display(prices_df.head())

Descargando EC -> EC
Columnas descargadas: ['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
Descargando ARG -> GRUPOARGOS.CL
Columnas descargadas: ['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
Descargando CIBEST -> CIBEST.CL
Columnas descargadas: ['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
Descargando NCH -> NUTRESA.CL
Columnas descargadas: ['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']

Precios guardados en: /content/drive/MyDrive/EAFIT/TESIS/precios_colombia.csv
Shape precios: (1962, 12)


,fecha,stock,open,high,low,close,volume,return_1d,volatility_5,ma_5,ma_10,target
9,2024-01-12,ARG,13700.0,13900.0,13700.0,12702.338867,291247,-0.005764,0.008953,12717.066406,12356.099512,1
10,2024-01-15,ARG,13900.0,13900.0,13700.0,12794.385742,159203,0.007246,0.009234,12753.885156,12504.872266,0
11,2024-01-16,ARG,13620.0,14120.0,13600.0,12628.702148,393345,-0.012950,0.008742,12720.748438,12593.379590,1
12,2024-01-17,ARG,13860.0,13960.0,13720.0,12683.930664,208865,0.004373,0.008744,12717.066797,12669.202539,0
13,2024-01-18,ARG,13800.0,13880.0,13540.0,12463.021484,283493,-0.017416,0.010674,12654.475781,12676.566406,0


In [ ]:
# BLOQUE 4. RECOLECCIÓN DE NOTICIAS

def fetch_google_news_entries_for_query(stock, query, start_date, end_date, step_days=7):
    rows = []
    chunks = chunk_date_ranges(start_date, end_date, step_days=step_days)

    for d1, d2 in tqdm(chunks, desc=f"{stock} | {query}", leave=False):
        rss_url = build_google_news_rss_url(query, d1, d2)
        feed = feedparser.parse(rss_url)

        for entry in feed.entries:
            published = None
            if hasattr(entry, "published_parsed") and entry.published_parsed:
                published = datetime(*entry.published_parsed[:6]).strftime("%Y-%m-%d")

            rows.append({
                "stock": stock,
                "query": query,
                "published_feed": published,
                "title_feed": normalize_text(entry.get("title", "")),
                "summary_feed": normalize_text(entry.get("summary", "")),
                "link_google_news": entry.get("link", "")
            })

        time.sleep(0.4)

    return rows

all_news_rows = []

for stock, queries in STOCK_QUERIES.items():
    for q in queries:
        tmp_rows = fetch_google_news_entries_for_query(
            stock=stock,
            query=q,
            start_date=START_DATE,
            end_date=END_DATE,
            step_days=7
        )
        all_news_rows.extend(tmp_rows)

news_df = pd.DataFrame(all_news_rows).drop_duplicates().reset_index(drop=True)

print("Noticias RSS encontradas:", news_df.shape)

# Resolver URL final
news_df["final_url"] = [
    resolve_final_url(u) for u in tqdm(news_df["link_google_news"].fillna("").tolist(), desc="Resolviendo URLs")
]
news_df["domain"] = news_df["final_url"].apply(get_domain)
news_df["is_priority_source"] = news_df["final_url"].apply(is_priority_source)

display(news_df.head())
print(news_df["domain"].value_counts().head(20))

Noticias RSS encontradas: (6543, 6)


Resolviendo URLs:  94%|█████████▍| 6156/6543 [8:19:23<47:37,  7.38s/it]

In [ ]:
# BLOQUE 5. EXTRACCIÓN, LIMPIEZA Y SENTIMIENTO

parsed_rows = []

for row in tqdm(news_df.to_dict(orient="records"), desc="Extrayendo artículos"):
    parsed = parse_article(row["final_url"])

    title_final = parsed["title_extracted"] if parsed["title_extracted"] else row["title_feed"]
    text_final = parsed["text_extracted"]
    publish_final = parsed["publish_date_extracted"] if parsed["publish_date_extracted"] else row["published_feed"]

    parsed_rows.append({
        "title_final": title_final,
        "text_final": text_final,
        "publish_final": publish_final
    })

parsed_df = pd.DataFrame(parsed_rows)
news_df = pd.concat([news_df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

# Relevancia
news_df["is_relevant"] = news_df.apply(
    lambda r: relevance_filter(
        r["stock"],
        r.get("title_final", ""),
        r.get("summary_feed", ""),
        r.get("text_final", "")
    ),
    axis=1
)

news_df = news_df[news_df["is_relevant"]].copy()

# Fecha final
news_df["fecha"] = pd.to_datetime(news_df["publish_final"], errors="coerce")
news_df["fecha"] = news_df["fecha"].fillna(pd.to_datetime(news_df["published_feed"], errors="coerce"))
news_df = news_df.dropna(subset=["fecha"]).copy()
news_df["fecha"] = news_df["fecha"].dt.strftime("%Y-%m-%d")

# Texto para sentimiento
news_df["text_for_sentiment"] = (
    news_df["title_final"].fillna("") + ". " +
    news_df["summary_feed"].fillna("") + ". " +
    news_df["text_final"].fillna("")
).str.strip()

# Duplicados
news_df = news_df.drop_duplicates(subset=["stock", "fecha", "title_final", "final_url"]).reset_index(drop=True)

# Sentimiento
labels = []
scores = []

for text in tqdm(news_df["text_for_sentiment"].fillna("").tolist(), desc="Sentimiento"):
    label, score = classify_sentiment(text)
    labels.append(label)
    scores.append(score)

news_df["sentiment_label"] = labels
news_df["sentiment_score"] = scores

news_df.to_csv(NEWS_RAW_PATH, index=False, encoding="utf-8-sig")

print("Noticias limpias guardadas en:", NEWS_RAW_PATH)
print("Shape noticias limpias:", news_df.shape)
display(news_df.head())